In [13]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
import spacy

# Imports
import pandas as pd
import numpy as np
import re
import string
from tqdm import tqdm

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import spacy

# Load spaCy model (CPU)
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ravid\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ravid\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [14]:
# load dataset
df = pd.read_csv("job_descriptions.csv")

print("Shape:", df.shape)
print("Columns:", df.columns)

df.head()

Shape: (1615940, 23)
Columns: Index(['Job Id', 'Experience', 'Qualifications', 'Salary Range', 'location',
       'Country', 'latitude', 'longitude', 'Work Type', 'Company Size',
       'Job Posting Date', 'Preference', 'Contact Person', 'Contact',
       'Job Title', 'Role', 'Job Portal', 'Job Description', 'Benefits',
       'skills', 'Responsibilities', 'Company', 'Company Profile'],
      dtype='object')


,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,...,Contact,Job Title,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile
0,1089843540111562,5 to 15 Years,M.Tech,$59K-$99K,Douglas,Isle of Man,54.2361,-4.5481,Intern,26801,...,001-381-930-7517x737,Digital Marketing Specialist,Social Media Manager,Snagajob,Social Media Managers oversee an organizations...,"{'Flexible Spending Accounts (FSAs), Relocatio...","Social media platforms (e.g., Facebook, Twitte...","Manage and grow social media accounts, create ...",Icahn Enterprises,"{""Sector"":""Diversified"",""Industry"":""Diversifie..."
1,398454096642776,2 to 12 Years,BCA,$56K-$116K,Ashgabat,Turkmenistan,38.9697,59.5563,Intern,100340,...,461-509-4216,Web Developer,Frontend Web Developer,Idealist,Frontend Web Developers design and implement u...,"{'Health Insurance, Retirement Plans, Paid Tim...","HTML, CSS, JavaScript Frontend frameworks (e.g...","Design and code user interfaces for websites, ...",PNC Financial Services Group,"{""Sector"":""Financial Services"",""Industry"":""Com..."
2,481640072963533,0 to 12 Years,PhD,$61K-$104K,Macao,"Macao SAR, China",22.1987,113.5439,Temporary,84525,...,9687619505,Operations Manager,Quality Control Manager,Jobs2Careers,Quality Control Managers establish and enforce...,"{'Legal Assistance, Bonuses and Incentive Prog...",Quality control processes and methodologies St...,Establish and enforce quality control standard...,United Services Automobile Assn.,"{""Sector"":""Insurance"",""Industry"":""Insurance: P..."
3,688192671473044,4 to 11 Years,PhD,$65K-$91K,Porto-Novo,Benin,9.3077,2.3158,Full-Time,129896,...,+1-820-643-5431x47576,Network Engineer,Wireless Network Engineer,FlexJobs,"Wireless Network Engineers design, implement, ...","{'Transportation Benefits, Professional Develo...",Wireless network design and architecture Wi-Fi...,"Design, configure, and optimize wireless netwo...",Hess,"{""Sector"":""Energy"",""Industry"":""Mining, Crude-O..."
4,117057806156508,1 to 12 Years,MBA,$64K-$87K,Santiago,Chile,-35.6751,-71.5429,Intern,53944,...,343.975.4702x9340,Event Manager,Conference Manager,Jobs2Careers,A Conference Manager coordinates and manages c...,"{'Flexible Spending Accounts (FSAs), Relocatio...",Event planning Conference logistics Budget man...,Specialize in conference and convention planni...,Cairn Energy,"{""Sector"":""Energy"",""Industry"":""Energy - Oil & ..."


In [15]:
def preprocess_text(text):
    doc = nlp(text)
    
    tokens = []
    for token in doc:
        if token.text not in stop_words and token.is_alpha:
            tokens.append(token.lemma_)
    
    return " ".join(tokens)

In [16]:
def clean_text(text):
    import re
    import string
    import pandas as pd
    
    if pd.isnull(text):
        return ""
    
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [17]:
df.columns = df.columns.str.strip()

TEXT_COLUMN = "Job Description"

df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)

tqdm.pandas()

df['cleaned_text'] = df[TEXT_COLUMN].progress_apply(clean_text)
df['processed_text'] = df['cleaned_text'].progress_apply(preprocess_text)

df.head()

100%|██████████| 1615940/1615940 [52:32<00:00, 512.51it/s] 


,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,...,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile,cleaned_text,processed_text
0,1089843540111562,5 to 15 Years,M.Tech,$59K-$99K,Douglas,Isle of Man,54.2361,-4.5481,Intern,26801,...,Social Media Manager,Snagajob,Social Media Managers oversee an organizations...,"{'Flexible Spending Accounts (FSAs), Relocatio...","Social media platforms (e.g., Facebook, Twitte...","Manage and grow social media accounts, create ...",Icahn Enterprises,"{""Sector"":""Diversified"",""Industry"":""Diversifie...",social media managers oversee an organizations...,social medium manager oversee organization soc...
1,398454096642776,2 to 12 Years,BCA,$56K-$116K,Ashgabat,Turkmenistan,38.9697,59.5563,Intern,100340,...,Frontend Web Developer,Idealist,Frontend Web Developers design and implement u...,"{'Health Insurance, Retirement Plans, Paid Tim...","HTML, CSS, JavaScript Frontend frameworks (e.g...","Design and code user interfaces for websites, ...",PNC Financial Services Group,"{""Sector"":""Financial Services"",""Industry"":""Com...",frontend web developers design and implement u...,frontend web developer design implement user i...
2,481640072963533,0 to 12 Years,PhD,$61K-$104K,Macao,"Macao SAR, China",22.1987,113.5439,Temporary,84525,...,Quality Control Manager,Jobs2Careers,Quality Control Managers establish and enforce...,"{'Legal Assistance, Bonuses and Incentive Prog...",Quality control processes and methodologies St...,Establish and enforce quality control standard...,United Services Automobile Assn.,"{""Sector"":""Insurance"",""Industry"":""Insurance: P...",quality control managers establish and enforce...,quality control manager establish enforce qual...
3,688192671473044,4 to 11 Years,PhD,$65K-$91K,Porto-Novo,Benin,9.3077,2.3158,Full-Time,129896,...,Wireless Network Engineer,FlexJobs,"Wireless Network Engineers design, implement, ...","{'Transportation Benefits, Professional Develo...",Wireless network design and architecture Wi-Fi...,"Design, configure, and optimize wireless netwo...",Hess,"{""Sector"":""Energy"",""Industry"":""Mining, Crude-O...",wireless network engineers design implement an...,wireless network engineer design implement mai...
4,117057806156508,1 to 12 Years,MBA,$64K-$87K,Santiago,Chile,-35.6751,-71.5429,Intern,53944,...,Conference Manager,Jobs2Careers,A Conference Manager coordinates and manages c...,"{'Flexible Spending Accounts (FSAs), Relocatio...",Event planning Conference logistics Budget man...,Specialize in conference and convention planni...,Cairn Energy,"{""Sector"":""Energy"",""Industry"":""Energy - Oil & ...",a conference manager coordinates and manages c...,conference manager coordinate manage conferenc...


In [18]:
skills_list = [
    'python', 'java', 'c++', 'machine learning', 'deep learning',
    'nlp', 'data analysis', 'sql', 'excel',
    'tensorflow', 'pytorch', 'communication',
    'teamwork', 'leadership', 'cloud', 'aws'
]

def extract_skills(text):
    found = []
    for skill in skills_list:
        if skill in text:
            found.append(skill)
    return ", ".join(found)

df['skills'] = df['processed_text'].apply(extract_skills)

df[['processed_text', 'skills']].head()

,processed_text,skills
0,social medium manager oversee organization soc...,
1,frontend web developer design implement user i...,
2,quality control manager establish enforce qual...,
3,wireless network engineer design implement mai...,communication
4,conference manager coordinate manage conferenc...,


In [19]:
df.to_csv("clean_job_dataset.csv", index=False)

print("✅ Saved as clean_job_dataset.csv")

✅ Saved as clean_job_dataset.csv


In [20]:
df.sample(5)

,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,...,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile,cleaned_text,processed_text
1533916,1707222229521922,2 to 9 Years,B.Tech,$55K-$115K,Banjul,Gambia,13.4432,-15.3101,Contract,110322,...,Pediatric Nurse Practitioner,Idealist,Pediatric Nurse Practitioners specialize in pe...,"{'Casual Dress Code, Social and Recreational A...",,"Specialize in pediatric care, addressing the u...",Telstra Corporation,"{""Sector"":""Telecommunications"",""Industry"":""Tel...",pediatric nurse practitioners specialize in pe...,pediatric nurse practitioner specialize pediat...
195339,41245530119953,2 to 9 Years,MCA,$60K-$96K,Lusaka,Zambia,-13.1339,27.8493,Contract,49200,...,Adult Speech Therapist,The Muse,Adult Speech Therapists work with adults to ad...,"{'Health Insurance, Retirement Plans, Paid Tim...",communication,Focus on speech and language therapy for adult...,Cadila Healthcare,"{""Sector"":""Pharmaceuticals"",""Industry"":""Pharma...",adult speech therapists work with adults to ad...,adult speech therapist work adult address spee...
1071832,2530191340428697,3 to 11 Years,MBA,$61K-$126K,Andorra la Vella,Andorra,42.5063,1.5218,Part-Time,76295,...,Architectural Drafter,Snagajob,Architectural Drafters assist architects and e...,"{'Employee Assistance Programs (EAP), Tuition ...",,Prepare detailed architectural drawings and pl...,Fiserv,"{""Sector"":""Financial Technology"",""Industry"":""F...",architectural drafters assist architects and e...,architectural drafter assist architect enginee...
561074,1048676582947802,1 to 15 Years,B.Tech,$64K-$95K,Apia,Samoa,-13.7590,-172.1046,Full-Time,130858,...,Purchasing Coordinator,SimplyHired,Purchasing Coordinators manage procurement pro...,"{'Casual Dress Code, Social and Recreational A...",,"Coordinate purchasing activities, including ve...",Hewlett Packard Enterprise,"{""Sector"":""Technology"",""Industry"":""Computers, ...",purchasing coordinators manage procurement pro...,purchase coordinator manage procurement proces...
19023,76612323315463,5 to 13 Years,PhD,$59K-$101K,Vientiane,Lao PDR,19.8563,102.4955,Full-Time,101492,...,Territory Sales Manager,USAJOBS,Territory Sales Managers oversee a specific sa...,"{'Life and Disability Insurance, Stock Options...",,"Manage a sales territory, including developing...",Valero Energy,"{""Sector"":""Energy"",""Industry"":""Petroleum Refin...",territory sales managers oversee a specific sa...,territory sale manager oversee specific sale t...
